<h2> Nauczanie maszynowe/Ćwiczenia6

* 18.11.2025, Jadwiga Krząstek

Zagadnienia na dziś:
- Maszyna wektorów nośnych - liniowa i nieliniowa
- Rola kernela w SVM

<h4> Zadanie1 (1 pkt): Rozważ dane Medical‑Abstracts‑TC‑Corpus — zbiór abstraktów medycznych z 5 klasami (nowotwory, choroby układu pokarmowego, nerwowego, sercowo‑naczyniowego i „ogólne stany patologiczne”). W oparciu o metodę SVM zbuduj klasyfikator na reprezentacjach TF-IDF tych abstraktów. Poniżej pare uwag:
    
- zbiory: treningowy i testowy są już wydzielone
- potestuj różne hiperparametry (zarówno dla SVM jak i TF - IDF, np. max_features)
- finalny raport klasyfikacji na zbiorze testowym powinien wynosić co najmniej 50 \% (a najlepiej więcej) dla f1-score dla każdej kategorii.
- być może przydatne będzie dogenerowanie nowych danych na podstawie istniejących przykładów (balansowanie danych)

https://huggingface.co/datasets/TimSchopf/medical_abstracts

In [25]:
from sklearn import datasets
import numpy as np
#from mlxtend.plotting import plot_decision_regions
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from imblearn.over_sampling import RandomOverSampler

In [1]:
import pandas as pd

df = pd.read_csv("medical_tc_train.csv")
print(df.head())

   condition_label                                   medical_abstract
0                5  Tissue changes around loose prostheses. A cani...
1                1  Neuropeptide Y and neuron-specific enolase lev...
2                2  Sexually transmitted diseases of the colon, re...
3                1  Lipolytic factors associated with murine and h...
4                3  Does carotid restenosis predict an increased r...


In [11]:
train_df = pd.read_csv("medical_tc_train.csv")
test_df = pd.read_csv("medical_tc_test.csv")

print("\nLiczba rekordów train:", len(train_df))
print("Liczba rekordów test:", len(test_df))

# sprawdzamy klasy
print("\nRozkład klas w train:")
print(train_df["condition_label"].value_counts())



Liczba rekordów train: 11550
Liczba rekordów test: 2888

Rozkład klas w train:
condition_label
5    3844
1    2530
4    2441
3    1540
2    1195
Name: count, dtype: int64


In [10]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",   
        max_features=30000,    
        ngram_range=(1,2)        
    )),
    ("svm", LinearSVC(class_weight="balanced"))
])

#trening
pipeline.fit(train_df["medical_abstract"], train_df["condition_label"])

#test
pred = pipeline.predict(test_df["medical_abstract"])

print(classification_report(test_df["condition_label"], pred))


              precision    recall  f1-score   support

           1       0.64      0.67      0.65       633
           2       0.41      0.48      0.44       299
           3       0.47      0.49      0.48       385
           4       0.60      0.64      0.62       610
           5       0.44      0.38      0.41       961

    accuracy                           0.52      2888
   macro avg       0.51      0.53      0.52      2888
weighted avg       0.52      0.52      0.52      2888



#### Warto zastosować oversampling w celu zbalansowania danych

In [13]:
#oversampling + TFIDF + SVM
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=30000,
        ngram_range=(1,2) #wyrazenia zlożone z dwóch słów mogą być szczególnie charakterystyczne
    )),
    ("oversample", RandomOverSampler()), #tworzy kopie istniejących próbek
    ("svm", LinearSVC(class_weight="balanced"))
])

pipeline.fit(train_df["medical_abstract"], train_df["condition_label"])

pred = pipeline.predict(test_df["medical_abstract"])

print(classification_report(test_df["condition_label"], pred))


              precision    recall  f1-score   support

           1       0.63      0.64      0.64       633
           2       0.40      0.47      0.44       299
           3       0.47      0.49      0.48       385
           4       0.59      0.62      0.60       610
           5       0.42      0.37      0.39       961

    accuracy                           0.51      2888
   macro avg       0.50      0.52      0.51      2888
weighted avg       0.51      0.51      0.51      2888



#### Brak zauważalnej poprawy, teraz dobrze będzie przetestować kilka wartości. Zmiany max_features podobnie jak, ngramów (2 lub 3) nie wpływają znacząco na wynik. 

In [24]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=30000,
        ngram_range=(1,3) #wyrazenia zlożone z trzech słów mogą być szczególnie charakterystyczne
    )),
    ("oversample", RandomOverSampler()), #tworzy kopie istniejących próbek
    ("svm", LinearSVC(class_weight="balanced"))
])

pipeline.fit(train_df["medical_abstract"], train_df["condition_label"])

pred = pipeline.predict(test_df["medical_abstract"])

print(classification_report(test_df["condition_label"], pred))

              precision    recall  f1-score   support

           1       0.62      0.64      0.63       633
           2       0.41      0.48      0.44       299
           3       0.47      0.48      0.47       385
           4       0.58      0.63      0.61       610
           5       0.41      0.35      0.38       961

    accuracy                           0.51      2888
   macro avg       0.50      0.52      0.51      2888
weighted avg       0.50      0.51      0.50      2888



#### Pozostałe zmiany nie wprowadziły znacznej poprawy.

<h4> Zadanie2 (1+2 pkt): Rozważ sekwencje intronów i eksonów z ludzkiego chromosomu 1. 

- Wydziel zbiór treningowy i testowy
- Zaproponuj reprezentacje dla tych sekwencji (np. częstości wybranych k-merów, inne pomysły mile widziane), zastanów się czy warto normalizować/standaryzować dane
- Zbuduj klasyfikator w oparciu o SVM, na zbiorze testowym f1-score dla każdej kategorii powinno wynosić co najmniej 90 \%
- Spróbuj zinterpretować zbudowany model - wksaż które cechy sekwencji najbardziej decydują o tym, że dana sekwencja jest eksonem/intronem - np. częstości wybranych k-merów. Możesz wykorzystać algorytm RFE (https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html).

Druga część zadania (stanowiąca rozszerzenie pierwszej części) będzie polegać na stworzeniu modelu (w oparciu o SVM) który dla sekwencji składajacej się z intronów oraz eksonów rozpozna miejsca zmiany typu sekwencji genu - złożonej zarówno z intronów jak i eksonów EEEEEIIIIIIEEEEEIIII - czyli powie nam gdzie kończą/zaczynają się eksony. W tym przypadku możesz np. poruszać się oknem o zadanej długości i zastosować model z części I - być może warto narysować odpowiedni wykres. Inne pomysły bardzo mile widziane! (liczę na waszą kreatywność :)). Wyznaczenie dokładnego miejsca zmiany może być trudne, jako kryterium przyjmij, np. +-kilkanaście/kilkadziesiąt pozycji jako próg tolerancji. Zaproponuj odpowiednią fukcję kosztu/miarę oceny.

- To zadanie (zadanie2) proszę opracować także w formie raportu - 1 strona pdf. Na następnych zajęciach poprosze 2/3 osoby o przedstawienie swojego rozwiązania - opisu algorytmu, interpretacji modelu oraz ewentualnie kodu.

In [1]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from Bio import SeqIO
import numpy as np
from collections import Counter
from sklearn.svm import SVC

In [7]:
intron_fasta = "introns.txt"
exon_fasta   = "exons.txt"

introns = []
exons = []

for record in SeqIO.parse(intron_fasta, "fasta"):
    introns.append(str(record.seq))

for record in SeqIO.parse(exon_fasta, "fasta"):
    exons.append(str(record.seq))

print("Introns loaded:", len(introns))
print("Exons loaded:", len(exons))



Introns loaded: 261
Exons loaded: 400


In [67]:
df_introns = pd.DataFrame({"sequence": introns, "label": 0})
df_exons   = pd.DataFrame({"sequence": exons, "label": 1})

df = pd.concat([df_introns, df_exons], ignore_index=True)
print(df.head())

                                            sequence  label
0  GTGAGTGGCCCCAGCCCCCTCCATTTCCCCAGGACTGCACTTTATG...      0
1  GTGAGTGGCCCCAGCCCCCTCCATTTCCCCAGGACTGCACTTTATG...      0
2  GTGAGTATTCAGTATGTGTGTTTTCCCCCTTCGGTGGATAACTGAA...      0
3  GTGAGGGGCGTGTGGGCAGAGGGGGTTTGGGACAGCAGGGTTTGTG...      0
4  GTCAGTGTGCATTTGTTTAGTTTTGATGCTTTTCAAATTACATTAT...      0


#### Wypisanie przykładowych 5-merów

In [66]:
def kmers(seq, k=5): #tworzenie k-merów dla k=5
    return " ".join(seq[i:i+k] for i in range(len(seq)-k+1))

X_train_km = X_train.apply(lambda s: kmers(s, 5))
X_test_km  = X_test.apply(lambda s: kmers(s, 5))
print(X_train_km.iloc[0][:200]) #wypisanie kilku początkowych k-merów


ATGAA TGAAC GAACA AACAC ACACC CACCA ACCAA CCAAC CAACG AACGA ACGAT CGATG GATGC ATGCC TGCCA GCCAA CCAAG CAAGG AAGGA AGGAG GGAGT GAGTA AGTAT GTATC TATCT ATCTG TCTGG CTGGC TGGCC GGCCC GCCCG CCCGG CCGGA CG


In [64]:
df_introns = pd.DataFrame({"sequence": introns, "label": 0})
df_exons   = pd.DataFrame({"sequence": exons,   "label": 1})

df = pd.concat([df_introns, df_exons], ignore_index=True)

X = df["sequence"].str.upper().str.strip()
y = df["label"]

#podział
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

#generowanie k-merów
def kmers(seq, k=5):
    return " ".join(seq[i:i+k] for i in range(len(seq)-k+1))

X_train_km = X_train.apply(lambda s: kmers(s, 5))
X_test_km  = X_test.apply(lambda s: kmers(s, 5))

#TF-IDF
tfidf = TfidfVectorizer(
    analyzer="word",  #każdy k-mer jest "słowem"
    token_pattern=r"\S+",  #każde nie-puste słowo traktujemy jako token
    min_df=1,
    max_features=50000
)


X_train_vec = tfidf.fit_transform(X_train_km)
X_test_vec  = tfidf.transform(X_test_km)

print("TF-IDF OK, shape:", X_train_vec.shape)


TF-IDF OK, shape: (528, 1024)


In [65]:
clf = LinearSVC(
    C=1.0,
    class_weight="balanced", #w celu zbalansowania klas
    max_iter=5000
)

clf.fit(X_train_vec, y_train)
y_pred = clf.predict(X_test_vec)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.93      0.98      0.95        53
           1       0.99      0.95      0.97        80

    accuracy                           0.96       133
   macro avg       0.96      0.97      0.96       133
weighted avg       0.96      0.96      0.96       133



#### Wysokie wyniki f1 i dobre accuracy 
introns 0 \
exons 1

#### Wskażmy teraz kilka 5-merów, które potencjalnie mogą różnicować introny od eksonów.

In [88]:
svm = SVC(kernel='linear', C=1., random_state=0, class_weight="balanced") 
svm.fit(X_train_vec, y_train)

#pobór danych z TF-IDF
feature_names = np.array(tfidf.get_feature_names_out())

#parametry SVM, współczynniki (wagi) dla poszczególnych k-merów
weights = svm.coef_.toarray().flatten()

#top k-mery dla eksonów (największe dodatnie wagi)
top_exons_idx = weights.argsort()[-5:][::-1]
top_exons = feature_names[top_exons_idx]

#top k-mery dla intronów (największe ujemne wagi)
top_introns_idx = weights.argsort()[:5]
top_introns = feature_names[top_introns_idx]

print("Top 5 k-merów eksonów:", top_exons)
print("Top 5 k-merów intronów:", top_introns)

Top 5 k-merów eksonów: ['gaaga' 'cacca' 'cagca' 'gagga' 'ccaga']
Top 5 k-merów intronów: ['ttttt' 'gtgag' 'agggg' 'taggg' 'gtaag']


#### **Podsumowanie**: eksony mają motywy bogate w GC (np. "GCGCG", "CGGCG"). Częstym początkiem sekwencji jest "ATGG". Introny natomiast charakteryzuą się motywami bogatymi w A i T, np. sekwencje "TTAAT" lub "TACTA" i rozpoczęciem przez "GTGAG" (drugie miejsce wśród najpopularniejszych k-merów).

#### Część 2: Wykrywanie granic intron–ekson w długiej sekwencji genomowej

#### **Metoda**: podajmy pewną sekwencję intronów i eksonów, np. złożoną z 10 sekwencji EEIIEIEEII, pobierzemy je losowo z obu plików i zapiszemy na jakich pozycjach następuje zmiana między nimi, np. (E, 100), (I,400) itd. Potem sprawdzimy jak model rozpoznaje te pozycje zmiany z dokładnością +-100. Jeśli to się uda to wypisuje np. rozpoznano poprawnie 8/10 zmian.

In [10]:
import random

df_introns = pd.DataFrame({"sequence": introns, "label": 0})
df_exons   = pd.DataFrame({"sequence": exons, "label": 1})

df = pd.concat([df_introns, df_exons], ignore_index=True)

pattern = "EEIIEIEEII"   #można dowolnie zmienić
pattern_list = list(pattern)

#losowanie sekwencji
def sample_sequence(pattern, introns, exons):
    seq = ""
    boundaries = []   #lista (klasa, pozycja)
    
    current_pos = 0
    for p in pattern:
        if p == "E":
            piece = random.choice(exons)
        else:
            piece = random.choice(introns)

        seq += piece
        current_pos += len(piece)
        
        boundaries.append((p, current_pos)) #pozycja końca segmentu

    return seq, boundaries

full_seq, true_boundaries = sample_sequence(pattern_list, introns, exons)

print("Długość sekwencji:", len(full_seq))
print("Granice prawdziwe:")
for b in true_boundaries:
    print(b)


Długość sekwencji: 18404
Granice prawdziwe:
('E', 2910)
('E', 3576)
('I', 3578)
('I', 4450)
('E', 7552)
('I', 7672)
('E', 8140)
('E', 8941)
('I', 13689)
('I', 18404)


In [11]:
def sliding_windows(seq, window=200, step=50):  #predykcja granic, czyli miejsc potencjalnej zmiany introna na ekson i odwrotnie
    windows = []
    positions = []

    for i in range(0, len(seq)-window, step):
        windows.append(seq[i:i+window])
        positions.append(i + window//2)   #bierzemy środek okna

    return windows, positions


In [22]:
from sklearn.svm import SVC
svm = SVC(kernel='linear', C=1., random_state=0, class_weight="balanced") 
svm.fit(X_train_vec, y_train)


def kmers(seq, k=5): #tworzenie k-merów dla k=5
    return " ".join(seq[i:i+k] for i in range(len(seq)-k+1))

win_seqs, win_positions = sliding_windows(full_seq, window=200, step=50)

win_kmers = [kmers(s, 5) for s in win_seqs] #kmers to funkcja z pierwszej części
win_vec = tfidf.transform(win_kmers)        #używamy pipeline TF-IDF z części 1

#klasyfikacja okien
predicted_labels = svm.predict(win_vec)   #0=Intron, 1=Exon

def predicted_boundaries(labels, positions):
    boundaries = []
    for i in range(1, len(labels)):
        if labels[i] != labels[i-1]:
            boundaries.append(positions[i])
    return boundaries

predicted_changes = predicted_boundaries(predicted_labels, win_positions)
print("Predykowane granice:", predicted_changes)


Predykowane granice: [450, 500, 1100, 1150, 1200, 1350, 1700, 1850, 1900, 2000, 2350, 2450, 2650, 2800, 3650, 3950, 4150, 4550, 4650, 4700, 4800, 4850, 5050, 5500, 5750, 5800, 5900, 5950, 6800, 6850, 6900, 7150, 7550, 7750, 8100, 8150, 8650, 8800, 9000, 13250, 13300, 14550, 14650, 14850, 15000, 15900, 15950, 17300, 17400, 17650, 17700]


In [30]:
#porównanie predykcji z prawdą (tolerancja ±100)
def evaluate_boundaries(true_b, pred_b, tolerance=100):
    correct = 0
    
    for t_class, t_pos in true_b:
        for p in pred_b:
            if abs(t_pos - p) <= tolerance:
                correct += 1
                break
    
    return correct, len(true_b)

correct, total = evaluate_boundaries(true_boundaries, predicted_changes)
print(f"Rozpoznano poprawnie {correct}/{total} granic.")


Rozpoznano poprawnie 7/10 granic.


#### Oczywiście ilość rozpoznanych granic zależy od tolerancji/wielkości okna. Przy tolerancji 10 rozpoznano ich poprawnie 2/10.